<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BTC Spot Grid Trading Backtest

**Workflow:** Setup & Data → Excel Grid Logic → V0 Fixed Grid → V1 Dynamic Grid + Risk Controls → Comparison → Diagnostics → System Audit → GitHub Log

- **V0** = fixed arithmetic-grid baseline.
- **V1** = causal dynamic re-centering arithmetic grid with live-oriented risk controls.
- V1 uses fixed order size; **compounding is OFF**.
- Live exchange order submission is **OFF** in this notebook.
- `Manual_logic_checker.ipynb` is used for manual scenario checks.
- `tests/test_grid_trading.py` contains automated deterministic tests.


## 1. Setup & Data


In [ ]:
import os
import heapq
import bisect
import json
import base64

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/03.Trading/00.Live Trading'

SYMBOL = 'BTCUSDT'
START_DATE = '2024-01-01'
END_DATE = '2026-01-01'

BUY_FEE = 0.001
SELL_FEE = 0.001


def load_market_data(symbol, timeframe, data_dir):
    path = os.path.join(
        data_dir,
        f'{symbol}-{timeframe}-combined.csv',
    )

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    df = pd.read_csv(path)

    required_columns = {
        'open_time',
        'open',
        'high',
        'low',
        'close',
        'volume',
    }
    missing = required_columns.difference(df.columns)
    if missing:
        raise ValueError(
            f'Missing market-data columns: {sorted(missing)}'
        )

    df['open_time'] = pd.to_datetime(
        df['open_time'],
        utc=True,
    )

    numeric_columns = [
        'open',
        'high',
        'low',
        'close',
        'volume',
    ]
    df[numeric_columns] = df[numeric_columns].astype(float)

    return (
        df.drop_duplicates('open_time')
        .sort_values('open_time')
        .reset_index(drop=True)
    )


df_1m = load_market_data(
    SYMBOL,
    '1m',
    DATA_DIR,
)

start_ts = pd.Timestamp(
    START_DATE,
    tz='UTC',
)
end_ts = pd.Timestamp(
    END_DATE,
    tz='UTC',
)

df_1m = df_1m.loc[
    (df_1m['open_time'] >= start_ts)
    & (df_1m['open_time'] < end_ts)
].reset_index(drop=True)

if df_1m.empty:
    raise ValueError('No market data inside the selected period.')

print('===== DATA =====')
print(f'Rows   : {len(df_1m):,}')
print(
    'Period : '
    f'{df_1m.open_time.min()} '
    f'-> {df_1m.open_time.max()}'
)


## 2. Excel Grid Logic / Checkpoint

The KZM Excel workbook remains the source-of-truth checkpoint for the arithmetic-grid calculation. This section confirms that the Python formula still reproduces the known first-grid profit from the workbook.


In [ ]:
# KZM Excel checkpoint parameters
GRID_CAPITAL = 3000.0
GRID_CEILING = 8987.0
GRID_FLOOR = 1987.0
GRID_GAP = 70.0

EXPECTED_FIRST_PROFIT = 0.1750644398340242


def build_excel_grid_table(
    capital,
    ceiling,
    floor,
    gap,
    buy_fee=0.001,
    sell_fee=0.001,
):
    if capital <= 0:
        raise ValueError(
            'capital must be greater than 0.'
        )

    if ceiling <= floor:
        raise ValueError(
            'ceiling must be greater than floor.'
        )

    if gap <= 0:
        raise ValueError(
            'gap must be greater than 0.'
        )

    if not (0 <= buy_fee < 1):
        raise ValueError(
            'buy_fee must be in [0, 1).'
        )

    if not (0 <= sell_fee < 1):
        raise ValueError(
            'sell_fee must be in [0, 1).'
        )

    raw_levels = (
        ceiling - floor
    ) / gap

    if not np.isclose(
        raw_levels,
        round(raw_levels),
    ):
        raise ValueError(
            '(ceiling - floor) must be '
            'exactly divisible by gap.'
        )

    number_of_grids = int(
        round(raw_levels)
    )

    capital_per_grid = (
        capital / number_of_grids
    )

    buy_prices = (
        ceiling
        - gap
        * np.arange(
            1,
            number_of_grids + 1,
        )
    )

    sell_prices = (
        buy_prices + gap
    )

    gross_base_amount = (
        capital_per_grid
        / buy_prices
    )

    buy_fee_base = (
        gross_base_amount
        * buy_fee
    )

    base_amount = (
        gross_base_amount
        - buy_fee_base
    )

    gross_sell = (
        base_amount
        * sell_prices
    )

    sell_fee_quote = (
        gross_sell
        * sell_fee
    )

    net_sell = (
        gross_sell
        - sell_fee_quote
    )

    profit = (
        net_sell
        - capital_per_grid
    )

    return pd.DataFrame({
        'level': np.arange(
            1,
            number_of_grids + 1,
        ),
        'buy_price': buy_prices,
        'sell_price': sell_prices,
        'capital_per_level': capital_per_grid,
        'gross_base_amount': gross_base_amount,
        'buy_fee_base': buy_fee_base,
        'base_amount': base_amount,
        'gross_sell': gross_sell,
        'sell_fee_quote': sell_fee_quote,
        'net_sell': net_sell,
        'profit': profit,
    })


df_grid_excel = build_excel_grid_table(
    GRID_CAPITAL,
    GRID_CEILING,
    GRID_FLOOR,
    GRID_GAP,
    BUY_FEE,
    SELL_FEE,
)

excel_check = df_grid_excel.iloc[0]

assert np.isclose(
    excel_check['profit'],
    EXPECTED_FIRST_PROFIT,
    atol=1e-12,
)

print(
    'Excel checkpoint PASSED: '
    f"{excel_check['buy_price']:.0f} "
    f"-> {excel_check['sell_price']:.0f}, "
    f"profit={excel_check['profit']:.12f}"
)


## 3. V0 Fixed Grid Configuration

V0 remains the fixed-grid benchmark. Its boundaries are selected from the full historical period, so this boundary selection contains look-ahead bias and should not be treated as a live deployment rule.


In [ ]:
BACKTEST_CAPITAL = 3000.0
BACKTEST_GAP = 1000.0
PRICE_ROUNDING = 1000.0

historical_low = float(
    df_1m['low'].min()
)
historical_high = float(
    df_1m['high'].max()
)

BACKTEST_FLOOR = (
    np.floor(
        historical_low
        / PRICE_ROUNDING
    )
    * PRICE_ROUNDING
)

BACKTEST_CEILING = (
    np.ceil(
        historical_high
        / PRICE_ROUNDING
    )
    * PRICE_ROUNDING
)

raw_backtest_levels = (
    BACKTEST_CEILING
    - BACKTEST_FLOOR
) / BACKTEST_GAP

if not np.isclose(
    raw_backtest_levels,
    round(raw_backtest_levels),
):
    raise ValueError(
        'Backtest range must be '
        'divisible by BACKTEST_GAP.'
    )

NUMBER_OF_GRIDS = int(
    round(raw_backtest_levels)
)

CAPITAL_PER_LEVEL = (
    BACKTEST_CAPITAL
    / NUMBER_OF_GRIDS
)

df_grid_backtest = build_excel_grid_table(
    BACKTEST_CAPITAL,
    BACKTEST_CEILING,
    BACKTEST_FLOOR,
    BACKTEST_GAP,
    BUY_FEE,
    SELL_FEE,
)

print(
    '===== V0 FIXED GRID CONFIGURATION ====='
)
print(
    f'Historical Low   : '
    f'{historical_low:,.2f} USDT'
)
print(
    f'Historical High  : '
    f'{historical_high:,.2f} USDT'
)
print(
    f'Grid Floor       : '
    f'{BACKTEST_FLOOR:,.2f} USDT'
)
print(
    f'Grid Ceiling     : '
    f'{BACKTEST_CEILING:,.2f} USDT'
)
print(
    f'Grid Gap         : '
    f'{BACKTEST_GAP:,.2f} USDT'
)
print(
    f'Number of Grids  : '
    f'{NUMBER_OF_GRIDS}'
)
print(
    f'Capital / Grid   : '
    f'{CAPITAL_PER_LEVEL:,.6f} USDT'
)


## 4. V0 Fixed Grid Engine — 1 Minute

Execution rules:

- BUY only on a downward crossing of a grid level.
- Existing SELL targets can fill when the candle High reaches the target.
- A newly opened BUY cannot SELL in the same candle.
- A grid sold in the current candle cannot rebuy in the same candle.
- Same-candle SELL proceeds are not reused for BUYs.
- BUY fee is deducted from BTC; SELL fee is deducted from USDT.


In [ ]:
def calculate_performance_statistics(
    data,
    equity_values,
    initial_capital,
):
    running_peak = np.maximum.accumulate(
        equity_values
    )

    drawdown = (
        equity_values
        / running_peak
        - 1.0
    )

    max_drawdown = float(
        drawdown.min()
    )

    final_equity = float(
        equity_values[-1]
    )

    net_return = (
        final_equity
        / initial_capital
        - 1.0
    )

    elapsed_days = (
        data['open_time'].iloc[-1]
        - data['open_time'].iloc[0]
    ).total_seconds() / 86400.0

    annualized_return = np.nan

    if (
        elapsed_days > 0
        and final_equity > 0
    ):
        annualized_log_growth = (
            np.log(
                final_equity
                / initial_capital
            )
            * (
                365.25
                / elapsed_days
            )
        )

        if annualized_log_growth < 700:
            annualized_return = float(
                np.expm1(
                    annualized_log_growth
                )
            )

    calmar_ratio = np.nan

    if (
        max_drawdown < 0
        and np.isfinite(
            annualized_return
        )
    ):
        calmar_ratio = float(
            annualized_return
            / abs(max_drawdown)
        )

    return {
        'final_equity': final_equity,
        'net_return': float(net_return),
        'annualized_return': annualized_return,
        'max_drawdown': max_drawdown,
        'calmar_ratio': calmar_ratio,
        'drawdown': drawdown,
    }


def run_grid_backtest(
    df_price,
    grid_table,
    initial_capital,
):
    required = {
        'open_time',
        'open',
        'high',
        'low',
        'close',
    }

    missing = required.difference(
        df_price.columns
    )

    if missing:
        raise ValueError(
            f'Missing columns: {sorted(missing)}'
        )

    if df_price.empty:
        raise ValueError(
            'df_price is empty.'
        )

    if initial_capital <= 0:
        raise ValueError(
            'initial_capital must be '
            'greater than 0.'
        )

    data = (
        df_price
        .sort_values('open_time')
        .reset_index(drop=True)
    )

    grid = (
        grid_table
        .sort_values('buy_price')
        .reset_index(drop=True)
        .copy()
    )

    if grid.empty:
        raise ValueError(
            'grid_table is empty.'
        )

    buy_prices = (
        grid['buy_price']
        .to_numpy(float)
    )

    sell_prices = (
        grid['sell_price']
        .to_numpy(float)
    )

    capital_per_level = (
        grid['capital_per_level']
        .to_numpy(float)
    )

    base_amount = (
        grid['base_amount']
        .to_numpy(float)
    )

    buy_fee_base = (
        grid['buy_fee_base']
        .to_numpy(float)
    )

    sell_fee_quote = (
        grid['sell_fee_quote']
        .to_numpy(float)
    )

    net_sell = (
        grid['net_sell']
        .to_numpy(float)
    )

    cycle_profit = (
        grid['profit']
        .to_numpy(float)
    )

    excel_level = (
        grid['level']
        .to_numpy(int)
    )

    if (
        len(grid) > 1
        and not np.allclose(
            np.diff(buy_prices),
            np.diff(buy_prices)[0],
        )
    ):
        raise ValueError(
            'Arithmetic grid required.'
        )

    holding = np.zeros(
        len(grid),
        dtype=bool,
    )

    buy_time = [
        None
    ] * len(grid)

    sell_heap = []

    cash = float(
        initial_capital
    )
    open_btc = 0.0

    realized_profit = 0.0

    total_buy_fee_btc = 0.0
    total_buy_fee_usdt_equiv = 0.0
    total_sell_fee_usdt = 0.0

    completed_cycles = 0

    trade_events = []
    completed_trades = []

    number_of_rows = len(data)

    equity_values = np.empty(
        number_of_rows
    )
    cash_values = np.empty(
        number_of_rows
    )
    btc_values = np.empty(
        number_of_rows
    )

    buy_price_list = (
        buy_prices.tolist()
    )

    prev_close = None
    event_id = 0

    for i, row in enumerate(
        data.itertuples(index=False)
    ):
        timestamp = row.open_time
        open_price = float(row.open)
        high_price = float(row.high)
        low_price = float(row.low)
        close_price = float(row.close)

        cash_at_candle_start = cash

        sold_this_candle = set()

        # 1) Existing SELL orders.
        while (
            sell_heap
            and sell_heap[0][0]
            <= high_price
        ):
            _, grid_index = heapq.heappop(
                sell_heap
            )

            if not holding[grid_index]:
                continue

            cash_before = cash
            btc_before = open_btc

            holding[grid_index] = False

            cash += net_sell[
                grid_index
            ]

            open_btc -= base_amount[
                grid_index
            ]

            if abs(open_btc) < 1e-12:
                open_btc = 0.0

            realized_profit += (
                cycle_profit[
                    grid_index
                ]
            )

            total_sell_fee_usdt += (
                sell_fee_quote[
                    grid_index
                ]
            )

            completed_cycles += 1

            sold_this_candle.add(
                grid_index
            )

            event_id += 1

            completed_trades.append({
                'grid_level': int(
                    excel_level[
                        grid_index
                    ]
                ),
                'buy_time': buy_time[
                    grid_index
                ],
                'sell_time': timestamp,
                'buy_price': buy_prices[
                    grid_index
                ],
                'sell_price': sell_prices[
                    grid_index
                ],
                'cost': capital_per_level[
                    grid_index
                ],
                'quote_cost': capital_per_level[
                    grid_index
                ],
                'base_amount': base_amount[
                    grid_index
                ],
                'actual_earn': net_sell[
                    grid_index
                ],
                'net_sell': net_sell[
                    grid_index
                ],
                'grid_cashflow': cycle_profit[
                    grid_index
                ],
                'profit': cycle_profit[
                    grid_index
                ],
            })

            trade_events.append({
                'event_id': event_id,
                'time': timestamp,
                'side': 'SELL',
                'grid_level': int(
                    excel_level[
                        grid_index
                    ]
                ),
                'price': sell_prices[
                    grid_index
                ],
                'base_amount': base_amount[
                    grid_index
                ],
                'quote_amount': net_sell[
                    grid_index
                ],
                'fee_base': 0.0,
                'fee_quote': sell_fee_quote[
                    grid_index
                ],
                'realized_profit': cycle_profit[
                    grid_index
                ],
                'cash_movement': net_sell[
                    grid_index
                ],
                'grid_cashflow': cycle_profit[
                    grid_index
                ],
                'cash_before': cash_before,
                'cash_after': cash,
                'btc_before': btc_before,
                'btc_after': open_btc,
            })

            buy_time[
                grid_index
            ] = None

        # 2) Downward BUY crossings.
        # SELL proceeds from this candle
        # are intentionally unavailable.
        buy_budget = (
            cash_at_candle_start
        )

        down_start = (
            open_price
            if prev_close is None
            else max(
                prev_close,
                open_price,
            )
        )

        if low_price < down_start:
            first_index = (
                bisect.bisect_left(
                    buy_price_list,
                    low_price,
                )
            )

            stop_index = (
                bisect.bisect_left(
                    buy_price_list,
                    down_start,
                )
            )

            # Higher levels are crossed
            # first on the way down.
            for grid_index in range(
                stop_index - 1,
                first_index - 1,
                -1,
            ):
                if (
                    holding[
                        grid_index
                    ]
                    or grid_index
                    in sold_this_candle
                ):
                    continue

                cost = (
                    capital_per_level[
                        grid_index
                    ]
                )

                if (
                    buy_budget
                    + 1e-12
                    < cost
                ):
                    break

                cash_before = cash
                btc_before = open_btc

                holding[
                    grid_index
                ] = True

                buy_time[
                    grid_index
                ] = timestamp

                buy_budget -= cost
                cash -= cost

                open_btc += (
                    base_amount[
                        grid_index
                    ]
                )

                total_buy_fee_btc += (
                    buy_fee_base[
                        grid_index
                    ]
                )

                total_buy_fee_usdt_equiv += (
                    buy_fee_base[
                        grid_index
                    ]
                    * buy_prices[
                        grid_index
                    ]
                )

                heapq.heappush(
                    sell_heap,
                    (
                        sell_prices[
                            grid_index
                        ],
                        grid_index,
                    ),
                )

                event_id += 1

                trade_events.append({
                    'event_id': event_id,
                    'time': timestamp,
                    'side': 'BUY',
                    'grid_level': int(
                        excel_level[
                            grid_index
                        ]
                    ),
                    'price': buy_prices[
                        grid_index
                    ],
                    'base_amount': base_amount[
                        grid_index
                    ],
                    'quote_amount': cost,
                    'fee_base': buy_fee_base[
                        grid_index
                    ],
                    'fee_quote': 0.0,
                    'realized_profit': 0.0,
                    'cash_movement': -cost,
                    'grid_cashflow': 0.0,
                    'cash_before': cash_before,
                    'cash_after': cash,
                    'btc_before': btc_before,
                    'btc_after': open_btc,
                })

        # 3) Mark to market at Close.
        equity_values[i] = (
            cash
            + open_btc
            * close_price
        )

        cash_values[i] = cash
        btc_values[i] = open_btc

        prev_close = close_price

    performance = (
        calculate_performance_statistics(
            data,
            equity_values,
            initial_capital,
        )
    )

    equity_curve = pd.DataFrame({
        'open_time': (
            data['open_time']
            .to_numpy()
        ),
        'close': (
            data['close']
            .to_numpy(float)
        ),
        'cash': cash_values,
        'btc': btc_values,
        'equity': equity_values,
        'drawdown': (
            performance['drawdown']
        ),
    })

    trade_log = pd.DataFrame(
        trade_events
    )

    if not trade_log.empty:
        trade_log[
            'cumulative_cash_movement'
        ] = (
            trade_log[
                'cash_movement'
            ]
            .cumsum()
        )

        trade_log[
            'cumulative_grid_cashflow'
        ] = (
            trade_log[
                'grid_cashflow'
            ]
            .cumsum()
        )

    summary = {
        'initial_capital': float(
            initial_capital
        ),
        'final_equity': performance[
            'final_equity'
        ],
        'net_return': performance[
            'net_return'
        ],
        'annualized_return': performance[
            'annualized_return'
        ],
        'max_drawdown': performance[
            'max_drawdown'
        ],
        'calmar_ratio': performance[
            'calmar_ratio'
        ],
        'completed_cycles': int(
            completed_cycles
        ),
        'open_positions': int(
            holding.sum()
        ),
        'final_cash': float(
            cash
        ),
        'final_btc': float(
            open_btc
        ),
        'realized_profit': float(
            realized_profit
        ),
        'unrealized_pnl': float(
            performance[
                'final_equity'
            ]
            - initial_capital
            - realized_profit
        ),
        'buy_fee_btc': float(
            total_buy_fee_btc
        ),
        'buy_fee_usdt_equiv': float(
            total_buy_fee_usdt_equiv
        ),
        'sell_fee_usdt': float(
            total_sell_fee_usdt
        ),
        'total_fee_usdt_equiv': float(
            total_buy_fee_usdt_equiv
            + total_sell_fee_usdt
        ),
    }

    return {
        'summary': summary,
        'trade_log': trade_log,
        'completed_trades': pd.DataFrame(
            completed_trades
        ),
        'equity_curve': equity_curve,
        'grid_state': grid.assign(
            holding=holding,
            buy_time=buy_time,
        ),
    }


## 5. Run V0 Fixed Grid


In [ ]:
v0 = run_grid_backtest(
    df_1m,
    df_grid_backtest,
    BACKTEST_CAPITAL,
)

v0_summary = v0['summary']

print('===== V0 BACKTEST SUMMARY =====')
print(
    f"Initial Capital   : "
    f"{v0_summary['initial_capital']:,.2f} USDT"
)
print(
    f"Final Equity      : "
    f"{v0_summary['final_equity']:,.2f} USDT"
)
print(
    f"Net Return        : "
    f"{v0_summary['net_return']:.2%}"
)
print(
    f"Annualized Return : "
    f"{v0_summary['annualized_return']:.2%}"
)
print(
    f"Max Drawdown      : "
    f"{v0_summary['max_drawdown']:.2%}"
)
print(
    f"Calmar Ratio      : "
    f"{v0_summary['calmar_ratio']:.3f}"
)
print(
    f"Completed Cycles  : "
    f"{v0_summary['completed_cycles']:,}"
)
print(
    f"Open Positions    : "
    f"{v0_summary['open_positions']:,}"
)
print(
    f"Final Cash        : "
    f"{v0_summary['final_cash']:,.2f} USDT"
)
print(
    f"Final BTC         : "
    f"{v0_summary['final_btc']:.8f} BTC"
)
print(
    f"Realized Profit   : "
    f"{v0_summary['realized_profit']:,.2f} USDT"
)
print(
    f"Unrealized P&L    : "
    f"{v0_summary['unrealized_pnl']:,.2f} USDT"
)
print(
    f"Total Fees        : "
    f"{v0_summary['total_fee_usdt_equiv']:,.2f} USDT"
)


## 6. V0 Monthly Results


In [ ]:
def build_monthly_portfolio_pnl(
    equity_curve,
    initial_capital,
    start_date=None,
    end_date=None,
):
    equity = (
        equity_curve[
            [
                'open_time',
                'equity',
            ]
        ]
        .copy()
    )

    equity['open_time'] = (
        pd.to_datetime(
            equity['open_time'],
            utc=True,
        )
    )

    if start_date is not None:
        equity = equity.loc[
            equity['open_time']
            >= pd.Timestamp(
                start_date,
                tz='UTC',
            )
        ]

    if end_date is not None:
        equity = equity.loc[
            equity['open_time']
            < pd.Timestamp(
                end_date,
                tz='UTC',
            )
        ]

    equity['month'] = (
        equity['open_time']
        .dt.strftime('%Y-%m')
    )

    monthly = (
        equity
        .groupby(
            'month',
            as_index=False,
        )['equity']
        .last()
        .rename(
            columns={
                'equity':
                'ending_equity'
            }
        )
    )

    monthly[
        'beginning_equity'
    ] = (
        monthly[
            'ending_equity'
        ]
        .shift(1)
    )

    if len(monthly):
        monthly.loc[
            monthly.index[0],
            'beginning_equity',
        ] = initial_capital

    monthly[
        'portfolio_net_pnl'
    ] = (
        monthly[
            'ending_equity'
        ]
        - monthly[
            'beginning_equity'
        ]
    )

    monthly[
        'monthly_return'
    ] = (
        monthly[
            'portfolio_net_pnl'
        ]
        / monthly[
            'beginning_equity'
        ]
    )

    monthly[
        'cumulative_portfolio_pnl'
    ] = (
        monthly[
            'portfolio_net_pnl'
        ]
        .cumsum()
    )

    return monthly[
        [
            'month',
            'beginning_equity',
            'ending_equity',
            'portfolio_net_pnl',
            'monthly_return',
            'cumulative_portfolio_pnl',
        ]
    ]


df_v0_completed = (
    v0['completed_trades']
    .copy()
)

if not df_v0_completed.empty:
    df_v0_completed['month'] = (
        pd.to_datetime(
            df_v0_completed[
                'sell_time'
            ],
            utc=True,
        )
        .dt.strftime('%Y-%m')
    )

    df_v0_grid_cashflow_monthly = (
        df_v0_completed
        .groupby(
            'month',
            as_index=False,
        )
        .agg(
            monthly_grid_cashflow=(
                'grid_cashflow',
                'sum',
            ),
            completed_cycles=(
                'grid_cashflow',
                'size',
            ),
        )
    )

    df_v0_grid_cashflow_monthly[
        'cumulative_grid_cashflow'
    ] = (
        df_v0_grid_cashflow_monthly[
            'monthly_grid_cashflow'
        ]
        .cumsum()
    )

else:
    df_v0_grid_cashflow_monthly = (
        pd.DataFrame(
            columns=[
                'month',
                'monthly_grid_cashflow',
                'completed_cycles',
                'cumulative_grid_cashflow',
            ]
        )
    )


df_v0_monthly = (
    build_monthly_portfolio_pnl(
        v0['equity_curve'],
        BACKTEST_CAPITAL,
        START_DATE,
        END_DATE,
    )
)

print(
    '===== V0 MONTHLY GRID CASHFLOW ====='
)
display(
    df_v0_grid_cashflow_monthly
    .round(6)
)

print(
    '===== V0 MONTHLY PORTFOLIO P&L ====='
)
display(
    df_v0_monthly
    .round(6)
)


## 7. V1 Dynamic Grid + Risk Configuration

V1 remains a backtest / shadow-readiness strategy. Real exchange order submission is disabled.

Risk controls:

- Minimum USDT cash reserve = 25% of initial capital.
- Maximum open positions = 18.
- Maximum deployed position cost = 60% of initial capital.
- Maximum projected BTC entry exposure = 60% of initial capital.
- Drawdown circuit breaker = 15%; once triggered it latches and blocks future BUYs.
- Existing SELL targets remain active after a risk halt.
- Re-centering continues causally.


In [ ]:
V1_GRID_GAP = BACKTEST_GAP
V1_NUMBER_OF_GRIDS = 30
V1_RECENTER_TRIGGER_GRIDS = 5

V1_CAPITAL_PER_GRID = (
    BACKTEST_CAPITAL
    / V1_NUMBER_OF_GRIDS
)

V1_MIN_CASH_RESERVE = (
    BACKTEST_CAPITAL
    * 0.25
)

V1_MAX_OPEN_POSITIONS = 18

V1_MAX_DEPLOYED_CAPITAL = (
    BACKTEST_CAPITAL
    * 0.60
)

V1_MAX_ENTRY_BTC_EXPOSURE = (
    BACKTEST_CAPITAL
    * 0.60
)

V1_MAX_DRAWDOWN_STOP = 0.15

LIVE_EXECUTION_ENABLED = False
EXECUTION_MODE = (
    'BACKTEST / SHADOW READINESS'
)

print(
    '===== V1 DYNAMIC GRID + RISK CONFIGURATION ====='
)
print(
    f'Gap                    : '
    f'{V1_GRID_GAP:,.0f} USDT'
)
print(
    f'Active Grid Intervals  : '
    f'{V1_NUMBER_OF_GRIDS}'
)
print(
    f'Capital / Grid         : '
    f'{V1_CAPITAL_PER_GRID:,.2f} USDT'
)
print(
    f'Recenter Trigger       : '
    f'{V1_RECENTER_TRIGGER_GRIDS} grids '
    f'({V1_RECENTER_TRIGGER_GRIDS * V1_GRID_GAP:,.0f} USDT)'
)
print(
    f'Min Cash Reserve       : '
    f'{V1_MIN_CASH_RESERVE:,.2f} USDT'
)
print(
    f'Max Open Positions     : '
    f'{V1_MAX_OPEN_POSITIONS}'
)
print(
    f'Max Deployed Capital   : '
    f'{V1_MAX_DEPLOYED_CAPITAL:,.2f} USDT'
)
print(
    f'Max Entry BTC Exposure : '
    f'{V1_MAX_ENTRY_BTC_EXPOSURE:,.2f} USDT'
)
print(
    f'Drawdown BUY Halt      : '
    f'{V1_MAX_DRAWDOWN_STOP:.1%}'
)
print(
    f'Execution Mode         : '
    f'{EXECUTION_MODE}'
)
print(
    f'Live Orders Enabled    : '
    f'{LIVE_EXECUTION_ENABLED}'
)


## 8. Dynamic Grid Helpers

The active arithmetic grid is centered around a causal reference price. For an even number of intervals, half of the BUY levels sit below the reference and half sit at/above the reference range. Existing positions are not modified when the regime re-centers.


In [ ]:
def round_to_gap(
    price,
    gap,
):
    if gap <= 0:
        raise ValueError(
            'gap must be greater than 0.'
        )

    return float(
        np.floor(
            float(price)
            / gap
            + 0.5
        )
        * gap
    )


def build_dynamic_regime(
    reference_price,
    gap,
    number_of_grids,
    capital,
    regime_id=0,
):
    if capital <= 0:
        raise ValueError(
            'capital must be greater than 0.'
        )

    if gap <= 0:
        raise ValueError(
            'gap must be greater than 0.'
        )

    if number_of_grids < 2:
        raise ValueError(
            'number_of_grids must be '
            'at least 2.'
        )

    lower_grids = (
        number_of_grids // 2
    )

    upper_grids = (
        number_of_grids
        - lower_grids
    )

    floor = float(
        reference_price
        - lower_grids
        * gap
    )

    ceiling = float(
        reference_price
        + upper_grids
        * gap
    )

    if floor <= 0:
        raise ValueError(
            'Dynamic grid floor must '
            'stay above 0.'
        )

    buy_prices = np.arange(
        floor,
        ceiling,
        gap,
        dtype=float,
    )

    if (
        len(buy_prices)
        != number_of_grids
    ):
        raise AssertionError(
            'Dynamic regime grid count '
            'mismatch.'
        )

    return {
        'regime_id': int(
            regime_id
        ),
        'reference_price': float(
            reference_price
        ),
        'floor': floor,
        'ceiling': ceiling,
        'buy_prices': buy_prices,
        'capital_per_grid': float(
            capital
            / number_of_grids
        ),
        'gap': float(gap),
    }


## 9. V1 Dynamic Re-centering Grid Engine

Dynamic behavior:

- Initial reference uses the first candle Open only.
- Re-center decision uses the current candle Close only.
- The new regime becomes effective from the next candle.
- Old positions retain their original SELL targets.

Risk behavior:

- Existing SELLs always remain active.
- Every new BUY must pass cash-reserve, open-position, deployed-capital, and projected BTC-entry-exposure checks.
- Drawdown halt is evaluated at candle Close and therefore affects BUYs from the next candle.
- Same-candle SELL proceeds are not reused for BUYs.


In [ ]:
def run_dynamic_grid_backtest(
    df_price,
    initial_capital,
    gap,
    number_of_grids,
    recenter_trigger_grids,
    buy_fee=0.001,
    sell_fee=0.001,
    min_cash_reserve=0.0,
    max_open_positions=None,
    max_deployed_capital=None,
    max_entry_btc_exposure=None,
    max_drawdown_stop=None,
):
    required = {
        'open_time',
        'open',
        'high',
        'low',
        'close',
    }

    missing = required.difference(
        df_price.columns
    )

    if missing:
        raise ValueError(
            f'Missing columns: {sorted(missing)}'
        )

    if df_price.empty:
        raise ValueError(
            'df_price is empty.'
        )

    if initial_capital <= 0:
        raise ValueError(
            'initial_capital must be '
            'greater than 0.'
        )

    if gap <= 0:
        raise ValueError(
            'gap must be greater than 0.'
        )

    if number_of_grids < 2:
        raise ValueError(
            'number_of_grids must be '
            'at least 2.'
        )

    if recenter_trigger_grids < 1:
        raise ValueError(
            'recenter_trigger_grids '
            'must be at least 1.'
        )

    if not (
        0 <= buy_fee < 1
        and 0 <= sell_fee < 1
    ):
        raise ValueError(
            'fees must be in [0, 1).'
        )

    if (
        min_cash_reserve < 0
        or min_cash_reserve
        > initial_capital
    ):
        raise ValueError(
            'min_cash_reserve must be '
            'in [0, initial_capital].'
        )

    if (
        max_open_positions
        is not None
        and max_open_positions < 1
    ):
        raise ValueError(
            'max_open_positions must '
            'be >= 1 or None.'
        )

    if (
        max_deployed_capital
        is not None
        and max_deployed_capital <= 0
    ):
        raise ValueError(
            'max_deployed_capital must '
            'be > 0 or None.'
        )

    if (
        max_entry_btc_exposure
        is not None
        and max_entry_btc_exposure <= 0
    ):
        raise ValueError(
            'max_entry_btc_exposure '
            'must be > 0 or None.'
        )

    if (
        max_drawdown_stop
        is not None
        and not (
            0
            < max_drawdown_stop
            <= 1
        )
    ):
        raise ValueError(
            'max_drawdown_stop must '
            'be in (0, 1] or None.'
        )

    data = (
        df_price
        .sort_values('open_time')
        .reset_index(drop=True)
    )

    initial_reference = (
        round_to_gap(
            float(
                data.iloc[0]['open']
            ),
            gap,
        )
    )

    regime_id = 0

    regime = build_dynamic_regime(
        initial_reference,
        gap,
        number_of_grids,
        initial_capital,
        regime_id,
    )

    regime_history = [{
        'regime_id': 0,
        'effective_time': (
            data.iloc[0][
                'open_time'
            ]
        ),
        'reference_price': (
            regime[
                'reference_price'
            ]
        ),
        'floor': regime['floor'],
        'ceiling': regime['ceiling'],
        'reason': 'INITIAL',
    }]

    recenter_events = []
    risk_halt_events = []

    cash = float(
        initial_capital
    )

    open_btc = 0.0
    deployed_capital = 0.0

    realized_profit = 0.0

    total_buy_fee_btc = 0.0
    total_buy_fee_usdt_equiv = 0.0
    total_sell_fee_usdt = 0.0

    completed_cycles = 0

    event_id = 0
    position_id = 0

    positions = {}
    open_by_buy_price = {}
    sell_heap = []

    trade_events = []
    completed_trades = []

    blocked_buy_counts = {
        'risk_halt': 0,
        'cash_reserve': 0,
        'max_open_positions': 0,
        'max_deployed_capital': 0,
        'max_entry_btc_exposure': 0,
    }

    number_of_rows = len(data)

    equity_values = np.empty(
        number_of_rows
    )
    cash_values = np.empty(
        number_of_rows
    )
    btc_values = np.empty(
        number_of_rows
    )
    deployed_values = np.empty(
        number_of_rows
    )
    open_position_values = np.empty(
        number_of_rows,
        dtype=int,
    )
    btc_market_value_values = np.empty(
        number_of_rows
    )
    reference_values = np.empty(
        number_of_rows
    )
    regime_values = np.empty(
        number_of_rows,
        dtype=int,
    )
    risk_halt_values = np.empty(
        number_of_rows,
        dtype=bool,
    )

    prev_close = None

    tolerance = 1e-9

    risk_halt = False

    peak_equity = float(
        initial_capital
    )

    max_entry_exposure_observed = 0.0

    for i, row in enumerate(
        data.itertuples(index=False)
    ):
        timestamp = row.open_time
        open_price = float(row.open)
        high_price = float(row.high)
        low_price = float(row.low)
        close_price = float(row.close)

        cash_at_candle_start = cash

        sold_this_candle = set()

        # 1) Existing SELL orders.
        while (
            sell_heap
            and sell_heap[0][0]
            <= high_price
            + tolerance
        ):
            _, current_position_id = (
                heapq.heappop(
                    sell_heap
                )
            )

            position = positions.get(
                current_position_id
            )

            if (
                position is None
                or not position[
                    'is_open'
                ]
            ):
                continue

            cash_before = cash
            btc_before = open_btc

            position[
                'is_open'
            ] = False

            cash += position[
                'net_sell'
            ]

            open_btc -= position[
                'base_amount'
            ]

            deployed_capital -= (
                position['cost']
            )

            if abs(open_btc) < 1e-12:
                open_btc = 0.0

            if (
                abs(
                    deployed_capital
                )
                < 1e-10
            ):
                deployed_capital = 0.0

            realized_profit += (
                position['profit']
            )

            total_sell_fee_usdt += (
                position[
                    'sell_fee_quote'
                ]
            )

            completed_cycles += 1

            sold_this_candle.add(
                position[
                    'buy_price'
                ]
            )

            open_by_buy_price.pop(
                position[
                    'buy_price'
                ],
                None,
            )

            event_id += 1

            completed_trades.append({
                'position_id': (
                    current_position_id
                ),
                'regime_id': (
                    position[
                        'regime_id'
                    ]
                ),
                'buy_time': (
                    position[
                        'buy_time'
                    ]
                ),
                'sell_time': timestamp,
                'buy_price': (
                    position[
                        'buy_price'
                    ]
                ),
                'sell_price': (
                    position[
                        'sell_price'
                    ]
                ),
                'cost': (
                    position['cost']
                ),
                'quote_cost': (
                    position['cost']
                ),
                'base_amount': (
                    position[
                        'base_amount'
                    ]
                ),
                'actual_earn': (
                    position[
                        'net_sell'
                    ]
                ),
                'net_sell': (
                    position[
                        'net_sell'
                    ]
                ),
                'grid_cashflow': (
                    position[
                        'profit'
                    ]
                ),
                'profit': (
                    position[
                        'profit'
                    ]
                ),
            })

            trade_events.append({
                'event_id': event_id,
                'time': timestamp,
                'side': 'SELL',
                'position_id': (
                    current_position_id
                ),
                'regime_id': (
                    position[
                        'regime_id'
                    ]
                ),
                'reference_price': (
                    position[
                        'reference_price'
                    ]
                ),
                'price': (
                    position[
                        'sell_price'
                    ]
                ),
                'base_amount': (
                    position[
                        'base_amount'
                    ]
                ),
                'quote_amount': (
                    position[
                        'net_sell'
                    ]
                ),
                'fee_base': 0.0,
                'fee_quote': (
                    position[
                        'sell_fee_quote'
                    ]
                ),
                'realized_profit': (
                    position[
                        'profit'
                    ]
                ),
                'cash_movement': (
                    position[
                        'net_sell'
                    ]
                ),
                'grid_cashflow': (
                    position[
                        'profit'
                    ]
                ),
                'cash_before': cash_before,
                'cash_after': cash,
                'btc_before': btc_before,
                'btc_after': open_btc,
                'deployed_capital_after': (
                    deployed_capital
                ),
                'entry_btc_exposure_after': (
                    np.nan
                ),
            })

        # 2) Downward BUY crossings.
        # SELL proceeds from this candle
        # are intentionally unavailable.
        buy_budget = (
            cash_at_candle_start
        )

        down_start = (
            open_price
            if prev_close is None
            else max(
                prev_close,
                open_price,
            )
        )

        buy_prices = (
            regime[
                'buy_prices'
            ]
        )

        buy_price_list = (
            buy_prices.tolist()
        )

        if low_price < down_start:
            first_index = (
                bisect.bisect_left(
                    buy_price_list,
                    low_price,
                )
            )

            stop_index = (
                bisect.bisect_left(
                    buy_price_list,
                    down_start,
                )
            )

            for grid_index in range(
                stop_index - 1,
                first_index - 1,
                -1,
            ):
                buy_price = float(
                    buy_prices[
                        grid_index
                    ]
                )

                if (
                    buy_price
                    in open_by_buy_price
                    or buy_price
                    in sold_this_candle
                ):
                    continue

                if risk_halt:
                    blocked_buy_counts[
                        'risk_halt'
                    ] += 1
                    break

                cost = float(
                    regime[
                        'capital_per_grid'
                    ]
                )

                if (
                    buy_budget
                    + tolerance
                    < cost
                ):
                    break

                if (
                    buy_budget
                    - cost
                    < min_cash_reserve
                    - tolerance
                ):
                    blocked_buy_counts[
                        'cash_reserve'
                    ] += 1
                    break

                current_open_positions = (
                    len(
                        open_by_buy_price
                    )
                )

                if (
                    max_open_positions
                    is not None
                    and current_open_positions
                    >= max_open_positions
                ):
                    blocked_buy_counts[
                        'max_open_positions'
                    ] += 1
                    break

                if (
                    max_deployed_capital
                    is not None
                    and deployed_capital
                    + cost
                    > max_deployed_capital
                    + tolerance
                ):
                    blocked_buy_counts[
                        'max_deployed_capital'
                    ] += 1
                    break

                sell_price = (
                    buy_price
                    + gap
                )

                gross_base_amount = (
                    cost
                    / buy_price
                )

                buy_fee_base = (
                    gross_base_amount
                    * buy_fee
                )

                base_amount = (
                    gross_base_amount
                    - buy_fee_base
                )

                projected_btc = (
                    open_btc
                    + base_amount
                )

                projected_entry_exposure = (
                    projected_btc
                    * buy_price
                )

                if (
                    max_entry_btc_exposure
                    is not None
                    and projected_entry_exposure
                    > max_entry_btc_exposure
                    + tolerance
                ):
                    blocked_buy_counts[
                        'max_entry_btc_exposure'
                    ] += 1
                    break

                gross_sell = (
                    base_amount
                    * sell_price
                )

                sell_fee_quote = (
                    gross_sell
                    * sell_fee
                )

                net_sell = (
                    gross_sell
                    - sell_fee_quote
                )

                cycle_profit = (
                    net_sell
                    - cost
                )

                cash_before = cash
                btc_before = open_btc

                buy_budget -= cost
                cash -= cost

                open_btc += (
                    base_amount
                )

                deployed_capital += (
                    cost
                )

                total_buy_fee_btc += (
                    buy_fee_base
                )

                total_buy_fee_usdt_equiv += (
                    buy_fee_base
                    * buy_price
                )

                max_entry_exposure_observed = max(
                    max_entry_exposure_observed,
                    projected_entry_exposure,
                )

                position_id += 1

                position = {
                    'position_id': (
                        position_id
                    ),
                    'regime_id': (
                        regime[
                            'regime_id'
                        ]
                    ),
                    'reference_price': (
                        regime[
                            'reference_price'
                        ]
                    ),
                    'buy_time': timestamp,
                    'buy_price': (
                        buy_price
                    ),
                    'sell_price': (
                        sell_price
                    ),
                    'cost': cost,
                    'base_amount': (
                        base_amount
                    ),
                    'buy_fee_base': (
                        buy_fee_base
                    ),
                    'sell_fee_quote': (
                        sell_fee_quote
                    ),
                    'net_sell': (
                        net_sell
                    ),
                    'profit': (
                        cycle_profit
                    ),
                    'is_open': True,
                }

                positions[
                    position_id
                ] = position

                open_by_buy_price[
                    buy_price
                ] = position_id

                heapq.heappush(
                    sell_heap,
                    (
                        sell_price,
                        position_id,
                    ),
                )

                event_id += 1

                trade_events.append({
                    'event_id': (
                        event_id
                    ),
                    'time': timestamp,
                    'side': 'BUY',
                    'position_id': (
                        position_id
                    ),
                    'regime_id': (
                        regime[
                            'regime_id'
                        ]
                    ),
                    'reference_price': (
                        regime[
                            'reference_price'
                        ]
                    ),
                    'price': (
                        buy_price
                    ),
                    'base_amount': (
                        base_amount
                    ),
                    'quote_amount': (
                        cost
                    ),
                    'fee_base': (
                        buy_fee_base
                    ),
                    'fee_quote': 0.0,
                    'realized_profit': 0.0,
                    'cash_movement': (
                        -cost
                    ),
                    'grid_cashflow': 0.0,
                    'cash_before': (
                        cash_before
                    ),
                    'cash_after': (
                        cash
                    ),
                    'btc_before': (
                        btc_before
                    ),
                    'btc_after': (
                        open_btc
                    ),
                    'deployed_capital_after': (
                        deployed_capital
                    ),
                    'entry_btc_exposure_after': (
                        projected_entry_exposure
                    ),
                })

        # 3) Mark to market at Close.
        equity = (
            cash
            + open_btc
            * close_price
        )

        peak_equity = max(
            peak_equity,
            equity,
        )

        current_drawdown = (
            equity
            / peak_equity
            - 1.0
        )

        # Drawdown decision is based on
        # this Close and is effective for
        # BUYs from the next candle.
        if (
            not risk_halt
            and max_drawdown_stop
            is not None
            and current_drawdown
            <= -max_drawdown_stop
        ):
            risk_halt = True

            risk_halt_events.append({
                'decision_time': (
                    timestamp
                ),
                'effective_time': (
                    data.iloc[
                        i + 1
                    ]['open_time']
                    if i + 1
                    < number_of_rows
                    else pd.NaT
                ),
                'drawdown': float(
                    current_drawdown
                ),
                'equity': float(
                    equity
                ),
                'peak_equity': float(
                    peak_equity
                ),
                'reason': (
                    'MAX_DRAWDOWN_BUY_HALT'
                ),
            })

        equity_values[i] = equity
        cash_values[i] = cash
        btc_values[i] = open_btc
        deployed_values[i] = (
            deployed_capital
        )
        open_position_values[i] = (
            len(
                open_by_buy_price
            )
        )
        btc_market_value_values[i] = (
            open_btc
            * close_price
        )
        reference_values[i] = (
            regime[
                'reference_price'
            ]
        )
        regime_values[i] = (
            regime[
                'regime_id'
            ]
        )
        risk_halt_values[i] = (
            risk_halt
        )

        # 4) Causal re-center.
        # Decision now; new regime is
        # used from the next candle.
        trigger_distance = (
            recenter_trigger_grids
            * gap
        )

        if (
            close_price
            >= regime[
                'reference_price'
            ]
            + trigger_distance
            or close_price
            <= regime[
                'reference_price'
            ]
            - trigger_distance
        ):
            new_reference = (
                round_to_gap(
                    close_price,
                    gap,
                )
            )

            if (
                new_reference
                != regime[
                    'reference_price'
                ]
            ):
                old_regime = regime

                regime_id += 1

                regime = (
                    build_dynamic_regime(
                        new_reference,
                        gap,
                        number_of_grids,
                        initial_capital,
                        regime_id,
                    )
                )

                direction = (
                    'UP'
                    if new_reference
                    > old_regime[
                        'reference_price'
                    ]
                    else 'DOWN'
                )

                recenter_events.append({
                    'decision_time': (
                        timestamp
                    ),
                    'direction': (
                        direction
                    ),
                    'close': (
                        close_price
                    ),
                    'old_regime_id': (
                        old_regime[
                            'regime_id'
                        ]
                    ),
                    'new_regime_id': (
                        regime_id
                    ),
                    'old_reference': (
                        old_regime[
                            'reference_price'
                        ]
                    ),
                    'new_reference': (
                        new_reference
                    ),
                    'old_floor': (
                        old_regime[
                            'floor'
                        ]
                    ),
                    'old_ceiling': (
                        old_regime[
                            'ceiling'
                        ]
                    ),
                    'new_floor': (
                        regime[
                            'floor'
                        ]
                    ),
                    'new_ceiling': (
                        regime[
                            'ceiling'
                        ]
                    ),
                })

                regime_history.append({
                    'regime_id': (
                        regime_id
                    ),
                    'effective_time': (
                        data.iloc[
                            i + 1
                        ]['open_time']
                        if i + 1
                        < number_of_rows
                        else pd.NaT
                    ),
                    'reference_price': (
                        regime[
                            'reference_price'
                        ]
                    ),
                    'floor': (
                        regime[
                            'floor'
                        ]
                    ),
                    'ceiling': (
                        regime[
                            'ceiling'
                        ]
                    ),
                    'reason': (
                        f'RECENTER_{direction}'
                    ),
                })

        prev_close = close_price

    performance = (
        calculate_performance_statistics(
            data,
            equity_values,
            initial_capital,
        )
    )

    equity_curve = pd.DataFrame({
        'open_time': (
            data['open_time']
            .to_numpy()
        ),
        'close': (
            data['close']
            .to_numpy(float)
        ),
        'cash': (
            cash_values
        ),
        'btc': (
            btc_values
        ),
        'btc_market_value': (
            btc_market_value_values
        ),
        'deployed_capital': (
            deployed_values
        ),
        'open_positions': (
            open_position_values
        ),
        'equity': (
            equity_values
        ),
        'reference_price': (
            reference_values
        ),
        'regime_id': (
            regime_values
        ),
        'risk_halt': (
            risk_halt_values
        ),
        'drawdown': (
            performance[
                'drawdown'
            ]
        ),
    })

    trade_log = pd.DataFrame(
        trade_events
    )

    if not trade_log.empty:
        trade_log[
            'cumulative_cash_movement'
        ] = (
            trade_log[
                'cash_movement'
            ]
            .cumsum()
        )

        trade_log[
            'cumulative_grid_cashflow'
        ] = (
            trade_log[
                'grid_cashflow'
            ]
            .cumsum()
        )

    open_positions = [
        position
        for position
        in positions.values()
        if position[
            'is_open'
        ]
    ]

    summary = {
        'initial_capital': float(
            initial_capital
        ),
        'final_equity': performance[
            'final_equity'
        ],
        'net_return': performance[
            'net_return'
        ],
        'annualized_return': performance[
            'annualized_return'
        ],
        'max_drawdown': performance[
            'max_drawdown'
        ],
        'calmar_ratio': performance[
            'calmar_ratio'
        ],
        'completed_cycles': int(
            completed_cycles
        ),
        'open_positions': int(
            len(
                open_positions
            )
        ),
        'final_cash': float(
            cash
        ),
        'final_btc': float(
            open_btc
        ),
        'final_deployed_capital': float(
            deployed_capital
        ),
        'realized_profit': float(
            realized_profit
        ),
        'unrealized_pnl': float(
            performance[
                'final_equity'
            ]
            - initial_capital
            - realized_profit
        ),
        'buy_fee_btc': float(
            total_buy_fee_btc
        ),
        'buy_fee_usdt_equiv': float(
            total_buy_fee_usdt_equiv
        ),
        'sell_fee_usdt': float(
            total_sell_fee_usdt
        ),
        'total_fee_usdt_equiv': float(
            total_buy_fee_usdt_equiv
            + total_sell_fee_usdt
        ),
        'recenter_count': int(
            len(
                recenter_events
            )
        ),
        'risk_halt_triggered': bool(
            risk_halt
        ),
        'risk_halt_count': int(
            len(
                risk_halt_events
            )
        ),
        'min_cash_observed': float(
            cash_values.min()
        ),
        'max_open_positions_observed': int(
            open_position_values.max()
        ),
        'max_deployed_capital_observed': float(
            deployed_values.max()
        ),
        'max_btc_market_value_observed': float(
            btc_market_value_values.max()
        ),
        'max_entry_btc_exposure_observed': float(
            max_entry_exposure_observed
        ),
        'blocked_buy_counts': {
            key: int(value)
            for key, value
            in blocked_buy_counts.items()
        },
    }

    return {
        'summary': summary,
        'trade_log': trade_log,
        'completed_trades': pd.DataFrame(
            completed_trades
        ),
        'equity_curve': equity_curve,
        'open_positions': pd.DataFrame(
            open_positions
        ),
        'recenter_log': pd.DataFrame(
            recenter_events
        ),
        'regime_history': pd.DataFrame(
            regime_history
        ),
        'risk_halt_log': pd.DataFrame(
            risk_halt_events
        ),
    }


## 10. Run V1 Dynamic Grid + Risk Controls


In [ ]:
v1 = run_dynamic_grid_backtest(
    df_1m,
    initial_capital=BACKTEST_CAPITAL,
    gap=V1_GRID_GAP,
    number_of_grids=V1_NUMBER_OF_GRIDS,
    recenter_trigger_grids=(
        V1_RECENTER_TRIGGER_GRIDS
    ),
    buy_fee=BUY_FEE,
    sell_fee=SELL_FEE,
    min_cash_reserve=(
        V1_MIN_CASH_RESERVE
    ),
    max_open_positions=(
        V1_MAX_OPEN_POSITIONS
    ),
    max_deployed_capital=(
        V1_MAX_DEPLOYED_CAPITAL
    ),
    max_entry_btc_exposure=(
        V1_MAX_ENTRY_BTC_EXPOSURE
    ),
    max_drawdown_stop=(
        V1_MAX_DRAWDOWN_STOP
    ),
)

v1_summary = v1['summary']

print(
    '===== V1 DYNAMIC GRID + RISK SUMMARY ====='
)
print(
    f"Initial Capital      : "
    f"{v1_summary['initial_capital']:,.2f} USDT"
)
print(
    f"Final Equity         : "
    f"{v1_summary['final_equity']:,.2f} USDT"
)
print(
    f"Net Return           : "
    f"{v1_summary['net_return']:.2%}"
)
print(
    f"Annualized Return    : "
    f"{v1_summary['annualized_return']:.2%}"
)
print(
    f"Max Drawdown         : "
    f"{v1_summary['max_drawdown']:.2%}"
)
print(
    f"Calmar Ratio         : "
    f"{v1_summary['calmar_ratio']:.3f}"
)
print(
    f"Completed Cycles     : "
    f"{v1_summary['completed_cycles']:,}"
)
print(
    f"Open Positions       : "
    f"{v1_summary['open_positions']:,}"
)
print(
    f"Recenter Count       : "
    f"{v1_summary['recenter_count']:,}"
)
print(
    f"Final Cash           : "
    f"{v1_summary['final_cash']:,.2f} USDT"
)
print(
    f"Final BTC            : "
    f"{v1_summary['final_btc']:.8f} BTC"
)
print(
    f"Deployed Capital     : "
    f"{v1_summary['final_deployed_capital']:,.2f} USDT"
)
print(
    f"Realized Profit      : "
    f"{v1_summary['realized_profit']:,.2f} USDT"
)
print(
    f"Unrealized P&L       : "
    f"{v1_summary['unrealized_pnl']:,.2f} USDT"
)
print(
    f"Total Fees           : "
    f"{v1_summary['total_fee_usdt_equiv']:,.2f} USDT"
)
print(
    f"Minimum Cash         : "
    f"{v1_summary['min_cash_observed']:,.2f} USDT"
)
print(
    f"Max Open Positions   : "
    f"{v1_summary['max_open_positions_observed']}"
)
print(
    f"Max Deployed Capital : "
    f"{v1_summary['max_deployed_capital_observed']:,.2f} USDT"
)
print(
    f"Risk Halt Triggered  : "
    f"{v1_summary['risk_halt_triggered']}"
)


## 11. V0 vs V1 Comparison

This is a strategy-level comparison. V0 and V1 do not use the same number of active grid intervals or order size, so the table should not be interpreted as a one-variable causal experiment.


In [ ]:
comparison_specs = [
    (
        'Final Equity (USDT)',
        'final_equity',
        1.0,
    ),
    (
        'Net Return (%)',
        'net_return',
        100.0,
    ),
    (
        'Annualized Return (%)',
        'annualized_return',
        100.0,
    ),
    (
        'Max Drawdown (%)',
        'max_drawdown',
        100.0,
    ),
    (
        'Calmar Ratio',
        'calmar_ratio',
        1.0,
    ),
    (
        'Completed Cycles',
        'completed_cycles',
        1.0,
    ),
    (
        'Open Positions',
        'open_positions',
        1.0,
    ),
    (
        'Realized Profit (USDT)',
        'realized_profit',
        1.0,
    ),
    (
        'Unrealized P&L (USDT)',
        'unrealized_pnl',
        1.0,
    ),
    (
        'Total Fees (USDT)',
        'total_fee_usdt_equiv',
        1.0,
    ),
    (
        'Final Cash (USDT)',
        'final_cash',
        1.0,
    ),
    (
        'Final BTC',
        'final_btc',
        1.0,
    ),
]

comparison_rows = []

for (
    label,
    key,
    scale,
) in comparison_specs:
    v0_value = (
        v0_summary[key]
        * scale
    )

    v1_value = (
        v1_summary[key]
        * scale
    )

    comparison_rows.append({
        'Metric': label,
        'V0 Fixed Grid': (
            v0_value
        ),
        'V1 Dynamic + Risk': (
            v1_value
        ),
        'Difference (V1-V0)': (
            v1_value
            - v0_value
        ),
    })

comparison_rows.extend([
    {
        'Metric': (
            'Recenter Count'
        ),
        'V0 Fixed Grid': 0,
        'V1 Dynamic + Risk': (
            v1_summary[
                'recenter_count'
            ]
        ),
        'Difference (V1-V0)': (
            v1_summary[
                'recenter_count'
            ]
        ),
    },
    {
        'Metric': (
            'Risk Halt Triggered'
        ),
        'V0 Fixed Grid': 0,
        'V1 Dynamic + Risk': int(
            v1_summary[
                'risk_halt_triggered'
            ]
        ),
        'Difference (V1-V0)': int(
            v1_summary[
                'risk_halt_triggered'
            ]
        ),
    },
])

df_comparison = pd.DataFrame(
    comparison_rows
)

display(
    df_comparison
    .round(6)
)


def monthly_strategy_result(
    result,
    initial_capital,
    prefix,
):
    monthly = (
        build_monthly_portfolio_pnl(
            result[
                'equity_curve'
            ],
            initial_capital,
            START_DATE,
            END_DATE,
        )
    )

    return monthly[
        [
            'month',
            'ending_equity',
            'portfolio_net_pnl',
        ]
    ].rename(
        columns={
            'ending_equity':
            f'{prefix}_ending_equity',
            'portfolio_net_pnl':
            f'{prefix}_net_pnl',
        }
    )


v0_monthly_compare = (
    monthly_strategy_result(
        v0,
        BACKTEST_CAPITAL,
        'v0',
    )
)

v1_monthly_compare = (
    monthly_strategy_result(
        v1,
        BACKTEST_CAPITAL,
        'v1',
    )
)

df_monthly_comparison = (
    v0_monthly_compare.merge(
        v1_monthly_compare,
        on='month',
    )
)

df_monthly_comparison[
    'equity_difference'
] = (
    df_monthly_comparison[
        'v1_ending_equity'
    ]
    - df_monthly_comparison[
        'v0_ending_equity'
    ]
)

df_monthly_comparison[
    'net_pnl_difference'
] = (
    df_monthly_comparison[
        'v1_net_pnl'
    ]
    - df_monthly_comparison[
        'v0_net_pnl'
    ]
)

print(
    '===== MONTHLY V0 vs V1 ====='
)

display(
    df_monthly_comparison
    .round(2)
)

plot_df = (
    df_monthly_comparison
    .copy()
)

plot_df[
    'month_date'
] = pd.to_datetime(
    plot_df['month']
)

plt.figure(
    figsize=(12, 5)
)

plt.plot(
    plot_df[
        'month_date'
    ],
    plot_df[
        'v0_ending_equity'
    ],
    label='V0 Fixed Grid',
)

plt.plot(
    plot_df[
        'month_date'
    ],
    plot_df[
        'v1_ending_equity'
    ],
    label='V1 Dynamic + Risk',
)

plt.title(
    'BTC Spot Grid — '
    'V0 vs V1 Month-End Equity'
)
plt.xlabel('Month')
plt.ylabel('Equity (USDT)')
plt.legend()
plt.grid(
    True,
    alpha=0.25,
)
plt.tight_layout()
plt.show()


## 12. V1 Recenter & Risk Diagnostics


In [ ]:
print(
    '===== V1 RECENTER DIAGNOSTICS ====='
)
print(
    f"Total re-centers: "
    f"{len(v1['recenter_log']):,}"
)

if not v1['recenter_log'].empty:
    print(
        'First 10 re-centers'
    )
    display(
        v1[
            'recenter_log'
        ].head(10)
    )

    print(
        'Last 10 re-centers'
    )
    display(
        v1[
            'recenter_log'
        ].tail(10)
    )

print(
    'Latest regime history'
)
display(
    v1[
        'regime_history'
    ].tail(10)
)

print()
print(
    '===== V1 RISK DIAGNOSTICS ====='
)
print(
    f"Minimum Cash Observed    : "
    f"{v1_summary['min_cash_observed']:,.2f} USDT"
)
print(
    f"Maximum Open Positions   : "
    f"{v1_summary['max_open_positions_observed']}"
)
print(
    f"Maximum Deployed Capital : "
    f"{v1_summary['max_deployed_capital_observed']:,.2f} USDT"
)
print(
    f"Max BTC Market Value     : "
    f"{v1_summary['max_btc_market_value_observed']:,.2f} USDT"
)
print(
    f"Max Entry BTC Exposure   : "
    f"{v1_summary['max_entry_btc_exposure_observed']:,.2f} USDT"
)
print(
    f"Risk Halt Triggered      : "
    f"{v1_summary['risk_halt_triggered']}"
)
print(
    f"Blocked BUY Counts       : "
    f"{v1_summary['blocked_buy_counts']}"
)

if not v1['risk_halt_log'].empty:
    print(
        'Risk-halt events'
    )
    display(
        v1[
            'risk_halt_log'
        ]
    )


## 13. System Audit

`System Audit` is intentionally kept in the main notebook because it checks full-run accounting and hard risk-control integrity over the complete historical dataset. Manual scenario checking remains separate in `Manual_logic_checker.ipynb`.


In [ ]:
def audit_strategy_accounting(
    result,
    initial_capital,
    strategy_name,
):
    summary = (
        result['summary']
    )

    trade_log = (
        result['trade_log']
    )

    equity_curve = (
        result['equity_curve']
    )

    cash_movement_sum = (
        trade_log[
            'cash_movement'
        ].sum()
        if not trade_log.empty
        else 0.0
    )

    grid_cashflow_sum = (
        trade_log[
            'grid_cashflow'
        ].sum()
        if not trade_log.empty
        else 0.0
    )

    cash_reconciliation_error = abs(
        initial_capital
        + cash_movement_sum
        - summary[
            'final_cash'
        ]
    )

    realized_profit_error = abs(
        grid_cashflow_sum
        - summary[
            'realized_profit'
        ]
    )

    equity_identity_error = float(
        np.max(
            np.abs(
                equity_curve[
                    'cash'
                ]
                + equity_curve[
                    'btc'
                ]
                * equity_curve[
                    'close'
                ]
                - equity_curve[
                    'equity'
                ]
            )
        )
    )

    minimum_cash = float(
        equity_curve[
            'cash'
        ].min()
    )

    return [
        {
            'Strategy': (
                strategy_name
            ),
            'Test': (
                'Cash reconciliation'
            ),
            'Expected': '<= 1e-8',
            'Actual': (
                cash_reconciliation_error
            ),
            'Status': (
                'PASS'
                if cash_reconciliation_error
                <= 1e-8
                else 'FAIL'
            ),
        },
        {
            'Strategy': (
                strategy_name
            ),
            'Test': (
                'Grid Cashflow = '
                'realized profit'
            ),
            'Expected': '<= 1e-8',
            'Actual': (
                realized_profit_error
            ),
            'Status': (
                'PASS'
                if realized_profit_error
                <= 1e-8
                else 'FAIL'
            ),
        },
        {
            'Strategy': (
                strategy_name
            ),
            'Test': (
                'Cash + BTC x Close '
                '= Equity'
            ),
            'Expected': '<= 1e-8',
            'Actual': (
                equity_identity_error
            ),
            'Status': (
                'PASS'
                if equity_identity_error
                <= 1e-8
                else 'FAIL'
            ),
        },
        {
            'Strategy': (
                strategy_name
            ),
            'Test': (
                'Cash never negative'
            ),
            'Expected': '>= 0',
            'Actual': (
                minimum_cash
            ),
            'Status': (
                'PASS'
                if minimum_cash
                >= -1e-8
                else 'FAIL'
            ),
        },
    ]


audit_rows = []

audit_rows.extend(
    audit_strategy_accounting(
        v0,
        BACKTEST_CAPITAL,
        'V0 Fixed Grid',
    )
)

audit_rows.extend(
    audit_strategy_accounting(
        v1,
        BACKTEST_CAPITAL,
        'V1 Dynamic + Risk',
    )
)

# Recenter threshold.
if not v1['recenter_log'].empty:
    recenter_threshold_ok = (
        (
            v1[
                'recenter_log'
            ]['close']
            - v1[
                'recenter_log'
            ]['old_reference']
        )
        .abs()
        + 1e-9
        >= (
            V1_RECENTER_TRIGGER_GRIDS
            * V1_GRID_GAP
        )
    ).all()

else:
    recenter_threshold_ok = True

audit_rows.append({
    'Strategy': (
        'V1 Dynamic + Risk'
    ),
    'Test': (
        'Recenter threshold respected'
    ),
    'Expected': 'True',
    'Actual': bool(
        recenter_threshold_ok
    ),
    'Status': (
        'PASS'
        if recenter_threshold_ok
        else 'FAIL'
    ),
})

# Hard risk controls.
risk_control_checks = [
    {
        'Test': (
            'Minimum cash reserve respected'
        ),
        'Expected': (
            f'>= {V1_MIN_CASH_RESERVE:.2f}'
        ),
        'Actual': (
            v1_summary[
                'min_cash_observed'
            ]
        ),
        'Passed': (
            v1_summary[
                'min_cash_observed'
            ]
            >= V1_MIN_CASH_RESERVE
            - 1e-8
        ),
    },
    {
        'Test': (
            'Maximum open positions respected'
        ),
        'Expected': (
            f'<= {V1_MAX_OPEN_POSITIONS}'
        ),
        'Actual': (
            v1_summary[
                'max_open_positions_observed'
            ]
        ),
        'Passed': (
            v1_summary[
                'max_open_positions_observed'
            ]
            <= V1_MAX_OPEN_POSITIONS
        ),
    },
    {
        'Test': (
            'Maximum deployed capital respected'
        ),
        'Expected': (
            f'<= {V1_MAX_DEPLOYED_CAPITAL:.2f}'
        ),
        'Actual': (
            v1_summary[
                'max_deployed_capital_observed'
            ]
        ),
        'Passed': (
            v1_summary[
                'max_deployed_capital_observed'
            ]
            <= V1_MAX_DEPLOYED_CAPITAL
            + 1e-8
        ),
    },
    {
        'Test': (
            'Maximum entry BTC exposure respected'
        ),
        'Expected': (
            f'<= {V1_MAX_ENTRY_BTC_EXPOSURE:.2f}'
        ),
        'Actual': (
            v1_summary[
                'max_entry_btc_exposure_observed'
            ]
        ),
        'Passed': (
            v1_summary[
                'max_entry_btc_exposure_observed'
            ]
            <= V1_MAX_ENTRY_BTC_EXPOSURE
            + 1e-8
        ),
    },
]

for check in risk_control_checks:
    audit_rows.append({
        'Strategy': (
            'V1 Dynamic + Risk'
        ),
        'Test': (
            check['Test']
        ),
        'Expected': (
            check['Expected']
        ),
        'Actual': (
            check['Actual']
        ),
        'Status': (
            'PASS'
            if check['Passed']
            else 'FAIL'
        ),
    })

# If risk halt triggered,
# no BUY may occur on/after its
# effective candle.
if not v1['risk_halt_log'].empty:
    first_halt_effective = (
        v1[
            'risk_halt_log'
        ]
        .iloc[0][
            'effective_time'
        ]
    )

    if pd.isna(
        first_halt_effective
    ):
        post_halt_buys = 0

    else:
        post_halt_buys = int(
            (
                v1[
                    'trade_log'
                ]['side']
                .eq('BUY')
                & (
                    pd.to_datetime(
                        v1[
                            'trade_log'
                        ]['time'],
                        utc=True,
                    )
                    >= pd.Timestamp(
                        first_halt_effective
                    )
                )
            )
            .sum()
        )

else:
    post_halt_buys = 0

audit_rows.append({
    'Strategy': (
        'V1 Dynamic + Risk'
    ),
    'Test': (
        'No BUY after '
        'drawdown risk halt'
    ),
    'Expected': '0',
    'Actual': (
        post_halt_buys
    ),
    'Status': (
        'PASS'
        if post_halt_buys == 0
        else 'FAIL'
    ),
})

# Monthly portfolio reconciliation.
monthly_results = {}

for (
    strategy_name,
    result,
) in [
    (
        'V0 Fixed Grid',
        v0,
    ),
    (
        'V1 Dynamic + Risk',
        v1,
    ),
]:
    monthly = (
        build_monthly_portfolio_pnl(
            result[
                'equity_curve'
            ],
            BACKTEST_CAPITAL,
            START_DATE,
            END_DATE,
        )
    )

    monthly_results[
        strategy_name
    ] = monthly

    portfolio_gain = (
        result[
            'summary'
        ]['final_equity']
        - BACKTEST_CAPITAL
    )

    monthly_error = abs(
        monthly[
            'portfolio_net_pnl'
        ].sum()
        - portfolio_gain
    )

    audit_rows.append({
        'Strategy': (
            strategy_name
        ),
        'Test': (
            'Monthly Portfolio Net P&L '
            '= total portfolio gain'
        ),
        'Expected': '<= 1e-8',
        'Actual': (
            monthly_error
        ),
        'Status': (
            'PASS'
            if monthly_error
            <= 1e-8
            else 'FAIL'
        ),
    })

df_system_test_log = (
    pd.DataFrame(
        audit_rows
    )
)

display(
    df_system_test_log
)

failed_system_tests = (
    df_system_test_log.loc[
        df_system_test_log[
            'Status'
        ]
        .ne('PASS')
    ]
)

SYSTEM_AUDIT_STATUS = (
    'PASS'
    if failed_system_tests.empty
    else 'FAIL'
)

print(
    f'SYSTEM AUDIT: '
    f'{SYSTEM_AUDIT_STATUS}'
)

if (
    SYSTEM_AUDIT_STATUS
    != 'PASS'
):
    raise AssertionError(
        'SYSTEM AUDIT FAILED'
    )


## 14. Results


In [ ]:
print(
    '===== FINAL STRATEGY COMPARISON ====='
)

display(
    df_comparison
    .round(6)
)

print()
print(
    'V0 Fixed Grid'
)
print(
    f"  Final Equity : "
    f"{v0_summary['final_equity']:,.2f} USDT"
)
print(
    f"  Net Return   : "
    f"{v0_summary['net_return']:.2%}"
)
print(
    f"  Max Drawdown : "
    f"{v0_summary['max_drawdown']:.2%}"
)
print(
    f"  Calmar       : "
    f"{v0_summary['calmar_ratio']:.3f}"
)

print()
print(
    'V1 Dynamic Grid + Risk Controls'
)
print(
    f"  Final Equity : "
    f"{v1_summary['final_equity']:,.2f} USDT"
)
print(
    f"  Net Return   : "
    f"{v1_summary['net_return']:.2%}"
)
print(
    f"  Max Drawdown : "
    f"{v1_summary['max_drawdown']:.2%}"
)
print(
    f"  Calmar       : "
    f"{v1_summary['calmar_ratio']:.3f}"
)
print(
    f"  Re-centers   : "
    f"{v1_summary['recenter_count']:,}"
)
print(
    f"  Risk Halt    : "
    f"{v1_summary['risk_halt_triggered']}"
)
print(
    f"  Min Cash     : "
    f"{v1_summary['min_cash_observed']:,.2f} USDT"
)
print(
    f"  Max Positions: "
    f"{v1_summary['max_open_positions_observed']}"
)

print()
print(
    f'System Audit  : '
    f'{SYSTEM_AUDIT_STATUS}'
)
print(
    f'Execution Mode: '
    f'{EXECUTION_MODE}'
)
print(
    f'Live Orders   : '
    f'{LIVE_EXECUTION_ENABLED}'
)


## 15. Export Latest Backtest Log to GitHub

The log stores the complete V0/V1 summaries, parameters, monthly comparison, re-center diagnostics, risk diagnostics, and System Audit results in `logs/latest_backtest_log.json`.


In [ ]:
def _json_safe(
    value,
):
    if isinstance(
        value,
        (np.integer,),
    ):
        return int(value)

    if isinstance(
        value,
        (np.floating,),
    ):
        if not np.isfinite(
            value
        ):
            return None
        return float(value)

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    if isinstance(
        value,
        pd.Period,
    ):
        return str(value)

    if pd.isna(value):
        return None

    return value


def _records(
    dataframe,
):
    return [
        {
            key: _json_safe(
                value
            )
            for key, value
            in row.items()
        }
        for row
        in dataframe.to_dict(
            orient='records'
        )
    ]


log_payload = {
    'log_schema_version': 5,
    'run_info': {
        'generated_at_utc': (
            pd.Timestamp.now(
                tz='UTC'
            )
            .isoformat()
        ),
        'repository': (
            'natdanaiii/Trading'
        ),
        'branch': 'main',
        'notebook': (
            'Grid_trading.ipynb'
        ),
        'symbol': SYMBOL,
        'start_date': START_DATE,
        'end_date': END_DATE,
        'timeframe': '1m',
        'data_rows': int(
            len(df_1m)
        ),
        'data_first_time': (
            df_1m[
                'open_time'
            ]
            .min()
            .isoformat()
        ),
        'data_last_time': (
            df_1m[
                'open_time'
            ]
            .max()
            .isoformat()
        ),
    },
    'common_parameters': {
        'initial_capital': (
            BACKTEST_CAPITAL
        ),
        'buy_fee': BUY_FEE,
        'sell_fee': SELL_FEE,
    },
    'v0_parameters': {
        'floor': (
            BACKTEST_FLOOR
        ),
        'ceiling': (
            BACKTEST_CEILING
        ),
        'gap': (
            BACKTEST_GAP
        ),
        'number_of_grids': (
            NUMBER_OF_GRIDS
        ),
        'capital_per_grid': (
            CAPITAL_PER_LEVEL
        ),
        'price_rounding': (
            PRICE_ROUNDING
        ),
        'historical_low': (
            historical_low
        ),
        'historical_high': (
            historical_high
        ),
        'look_ahead_boundary_selection': (
            True
        ),
    },
    'v1_parameters': {
        'gap': (
            V1_GRID_GAP
        ),
        'number_of_grids': (
            V1_NUMBER_OF_GRIDS
        ),
        'capital_per_grid': (
            V1_CAPITAL_PER_GRID
        ),
        'recenter_trigger_grids': (
            V1_RECENTER_TRIGGER_GRIDS
        ),
        'recenter_trigger_usdt': (
            V1_RECENTER_TRIGGER_GRIDS
            * V1_GRID_GAP
        ),
        'compounding': False,
        'causal_recenter': True,
        'min_cash_reserve': (
            V1_MIN_CASH_RESERVE
        ),
        'max_open_positions': (
            V1_MAX_OPEN_POSITIONS
        ),
        'max_deployed_capital': (
            V1_MAX_DEPLOYED_CAPITAL
        ),
        'max_entry_btc_exposure': (
            V1_MAX_ENTRY_BTC_EXPOSURE
        ),
        'max_drawdown_stop': (
            V1_MAX_DRAWDOWN_STOP
        ),
        'live_execution_enabled': (
            LIVE_EXECUTION_ENABLED
        ),
        'execution_mode': (
            EXECUTION_MODE
        ),
    },
    'system_audit': {
        'status': (
            SYSTEM_AUDIT_STATUS
        ),
        'checks': _records(
            df_system_test_log
        ),
    },
    'v0_summary': {
        key: _json_safe(value)
        for key, value
        in v0_summary.items()
    },
    'v1_summary': {
        key: (
            value
            if isinstance(
                value,
                dict,
            )
            else _json_safe(
                value
            )
        )
        for key, value
        in v1_summary.items()
    },
    'comparison': _records(
        df_comparison
    ),
    'monthly_comparison': _records(
        df_monthly_comparison
    ),
    'v0_monthly_grid_cashflow': (
        _records(
            df_v0_grid_cashflow_monthly
        )
    ),
    'recenter_diagnostics': {
        'count': int(
            len(
                v1[
                    'recenter_log'
                ]
            )
        ),
        'first_10': _records(
            v1[
                'recenter_log'
            ].head(10)
        ),
        'last_10': _records(
            v1[
                'recenter_log'
            ].tail(10)
        ),
        'latest_regimes': _records(
            v1[
                'regime_history'
            ].tail(10)
        ),
    },
    'risk_diagnostics': {
        'risk_halt_log': (
            _records(
                v1[
                    'risk_halt_log'
                ]
            )
        ),
        'min_cash_observed': (
            v1_summary[
                'min_cash_observed'
            ]
        ),
        'max_open_positions_observed': (
            v1_summary[
                'max_open_positions_observed'
            ]
        ),
        'max_deployed_capital_observed': (
            v1_summary[
                'max_deployed_capital_observed'
            ]
        ),
        'max_btc_market_value_observed': (
            v1_summary[
                'max_btc_market_value_observed'
            ]
        ),
        'max_entry_btc_exposure_observed': (
            v1_summary[
                'max_entry_btc_exposure_observed'
            ]
        ),
        'blocked_buy_counts': (
            v1_summary[
                'blocked_buy_counts'
            ]
        ),
    },
}

LOCAL_LOG_PATH = (
    '/content/'
    'latest_backtest_log.json'
)

with open(
    LOCAL_LOG_PATH,
    'w',
) as file:
    json.dump(
        log_payload,
        file,
        indent=2,
        allow_nan=False,
    )

print(
    '===== BACKTEST LOG ====='
)
print(
    f'Local log    : '
    f'{LOCAL_LOG_PATH}'
)
print(
    f'System Audit : '
    f'{SYSTEM_AUDIT_STATUS}'
)

try:
    from google.colab import userdata

    github_token = (
        userdata.get(
            'GITHUB_TOKEN'
        )
    )

except Exception:
    github_token = None

if not github_token:
    print()
    print(
        "GitHub upload SKIPPED: "
        "Colab Secret "
        "'GITHUB_TOKEN' "
        "was not found."
    )

else:
    import requests

    repository = (
        'natdanaiii/Trading'
    )

    path = (
        'logs/'
        'latest_backtest_log.json'
    )

    api_url = (
        'https://api.github.com/'
        f'repos/{repository}/'
        f'contents/{path}'
    )

    headers = {
        'Authorization': (
            f'Bearer {github_token}'
        ),
        'Accept': (
            'application/'
            'vnd.github+json'
        ),
        'X-GitHub-Api-Version': (
            '2022-11-28'
        ),
    }

    existing = requests.get(
        api_url,
        headers=headers,
        timeout=30,
    )

    existing_sha = (
        existing.json().get(
            'sha'
        )
        if existing.status_code
        == 200
        else None
    )

    encoded = base64.b64encode(
        json.dumps(
            log_payload,
            indent=2,
        )
        .encode()
    ).decode()

    body = {
        'message': (
            'Update latest '
            'risk-controlled '
            'backtest log'
        ),
        'content': encoded,
        'branch': 'main',
    }

    if existing_sha:
        body['sha'] = (
            existing_sha
        )

    upload = requests.put(
        api_url,
        headers=headers,
        json=body,
        timeout=30,
    )

    upload.raise_for_status()

    print()
    print(
        'GitHub upload : SUCCESS'
    )
    print(
        f'Path          : '
        f'{path}'
    )
    print(
        f"Commit SHA    : "
        f"{upload.json()['commit']['sha']}"
    )
